# LightGBM 回归 — 完整流程 Demo

本 notebook 演示使用 `ml_tool` 工具包完成 LightGBM 回归任务的完整流程：

1. 数据初始化与集合划分（train / test / OOT）
2. 特征分析（缺失率、一值率、分位数统计、PSI、相关性）
3. 特征筛选（回归模式：缺失率 + PSI + 相关性，跳过 IV/WOE）
4. 模型训练（Hyperopt 自动调参）
5. 自定义超参数搜索空间
6. 回归评估报告（MAE / RMSE / R² / 分桶分析）
7. 特征分析报告
8. 模型保存与加载验证

In [ ]:
import pandas as pd
import numpy as np
from ml_tool import FeatureAnalyzer, FeatureSelector, ModelTrainer, ReportGenerator, split_dataset

np.random.seed(42)
n = 3000
dates = pd.date_range('2024-01-01', periods=n, freq='D').to_series().sample(frac=1, random_state=42).values

f1 = np.random.randn(n)
f2 = np.random.rand(n)
f3 = np.random.randn(n) * 2
f4 = np.random.rand(n)
f5 = np.random.randn(n)
noise = np.random.randn(n) * 0.5
y_reg = 2 * f1 - f2 + noise

df_raw = pd.DataFrame({
    'f1': f1, 'f2': f2, 'f3': f3, 'f4': f4, 'f5': f5,
    'y_reg': y_reg,
    'report_date': dates,
})
df_raw.loc[df_raw.sample(frac=0.05, random_state=0).index, 'f2'] = np.nan

df = split_dataset(df_raw, date_col='report_date', oot_date='2031-01-01', train_ratio=0.8)
df['month'] = df['report_date'].dt.to_period('M').astype(str)
print(df.shape, df['dataset'].value_counts().sort_index().to_dict())
print('目标变量统计:')
print(df['y_reg'].describe())

OUTPUT_DIR = './output/lgbm_reg'
# 模拟附加列（在 split_dataset 之后追加）
df['score_A']    = np.random.uniform(300, 700, len(df))   # 模拟标品A分
df['score_B']    = np.random.uniform(350, 750, len(df))   # 模拟标品B分
df['gain_score'] = np.random.uniform(0, 1, len(df))       # 模拟外部增益分
df['overdue_30'] = np.random.binomial(1, 0.12, len(df))   # 模拟逾期30天标签
df['overdue_60'] = np.random.binomial(1, 0.08, len(df))   # 模拟逾期60天标签

## 1. 特征分析

In [ ]:
feature_cols = ['f1', 'f2', 'f3', 'f4', 'f5']
analyzer = FeatureAnalyzer(df[feature_cols])

print('=== 缺失率 ==='); display(analyzer.missing_rate())
print('=== 一值率 ==='); display(analyzer.single_value_rate())
print('=== 分位数统计 ==='); display(analyzer.quantile_stats())

base_df    = df[df['dataset'] == 'train'][feature_cols].copy()
compare_df = df[df['dataset'] == 'oot'][feature_cols].copy()
psi_df = analyzer.psi(base_df, compare_df)
print('=== PSI（train vs OOT）==='); display(psi_df)
print('=== 特征间相关性（|r|>0.3）==='); display(analyzer.correlation(threshold=0.3))
# 回归任务无 IV 计算


## 2. 特征筛选（回归：缺失率 + PSI + 相关性，不做 IV 筛选）

In [ ]:
import os

reporter = ReportGenerator(output_dir=OUTPUT_DIR)
train_df = df[df['dataset'] == 'train'].reset_index(drop=True)

# 回归任务：psi_filter=False 关闭自动 PSI 过滤，corr_filter=False 关闭自动相关性过滤
# iv_threshold=0.0 跳过 IV 筛选
selector = FeatureSelector(
    iv_threshold=0.0,
    psi_threshold=0.5,
    missing_threshold=0.98,
    corr_threshold=0.97,
    psi_filter=False,
    corr_filter=False,
)
selector.fit(
    df=train_df[feature_cols + ['y_reg']],
    target='y_reg',
    base_df=base_df,
    compare_df=compare_df,
)
print('入选特征:', selector.get_selected())
print('剔除特征:', selector.dropped_cols)
display(selector.get_report())

analysis = {
    '缺失率': analyzer.missing_rate(),
    '分位数统计': analyzer.quantile_stats(),
    'PSI': psi_df,
    '相关性': analyzer.correlation(threshold=0.0),
}
path = reporter.feature_selection_report(selector.get_report(), analysis_results=analysis)
print('特征筛选报告:', path)


## 3. 模型训练（LightGBM 回归）

In [ ]:
selected = selector.get_selected()

X_train = df[df['dataset'] == 'train'][selected].reset_index(drop=True)
y_train = df[df['dataset'] == 'train']['y_reg'].reset_index(drop=True)
X_test  = df[df['dataset'] == 'test'][selected].reset_index(drop=True)
y_test  = df[df['dataset'] == 'test']['y_reg'].reset_index(drop=True)
X_oot   = df[df['dataset'] == 'oot'][selected].reset_index(drop=True)
y_oot   = df[df['dataset'] == 'oot']['y_reg'].reset_index(drop=True)

# n_trials 实际建议 100，这里用 20 演示
trainer = ModelTrainer(model_type='lgbm_reg', n_trials=20)
trainer.fit(
    X_train, y_train,
    X_test,  y_test,
    X_oot,   y_oot,
    save_dir=OUTPUT_DIR,
)
print('入模特征数:', len(trainer.selected_features))
print('入模特征:', trainer.selected_features)
print('最优参数:', trainer.best_params)


In [ ]:
print('调参日志（前 10 行，按轮次/试验顺序）:')
display(
    trainer.trials_log[['round', 'trial', 'loss', 'is_best', 'train_mae', 'test_mae', 'oot_mae']]
    .head(10)
)


## 4. 自定义超参数空间（可选）

In [ ]:
# 查看默认固定参数
default_params = trainer.get_default_params(n_data=len(X_train))
print('默认固定参数:')
for k, val in default_params.items():
    print(f'  {k:25s}: {val}')

print()
# 查看默认搜索空间（返回普通 dict，可直接覆盖任意 key）
default_space = trainer.get_default_space(n_data=len(X_train))
print('默认搜索空间 key:')
for k, val in default_space.items():
    print(f'  {k:25s}: {val}')


In [ ]:
from hyperopt import hp

# 基于默认空间修改部分参数，限制树深度和学习率
custom_space = trainer.get_default_space(n_data=len(X_train))
custom_space['learning_rate']    = hp.uniform('learning_rate', 0.005, 0.02)
custom_space['max_depth']        = hp.quniform('max_depth', 2, 4, 1)
custom_space['num_leaves']       = hp.quniform('num_leaves', 4, 24, 4)
custom_space['n_estimators']     = hp.quniform('n_estimators', 100, 400, 50)
custom_space['bagging_fraction'] = 0.8   # 固定为常量，不参与搜索

# 自定义固定参数（可选，不传则使用默认）
custom_p1 = trainer.get_default_params(n_data=len(X_train))

trainer_custom = ModelTrainer(model_type='lgbm_reg', n_trials=10)
trainer_custom.fit(
    X_train, y_train, X_test, y_test, X_oot, y_oot,
    custom_params=custom_p1,
    custom_space=custom_space,
)
print('最终参数:')
for k, val in trainer_custom.best_params.items():
    print(f'  {k:25s}: {val}')


## 5. 模型评估报告

In [ ]:
# 特征重要性
fi_df = pd.DataFrame({
    'feature':    trainer._trainer.model.feature_name(),
    'importance': trainer._trainer.model.feature_importance('gain'),
})

month_data = {
    '训练集': df[df['dataset'] == 'train'][['month']].reset_index(drop=True),
    '验证集': df[df['dataset'] == 'test'][['month']].reset_index(drop=True),
    'OOT':   df[df['dataset'] == 'oot'][['month']].reset_index(drop=True),
}
datasets_reg = {
    '训练集': (y_train.values, trainer.predict(X_train)),
    '验证集': (y_test.values,  trainer.predict(X_test)),
    'OOT':   (y_oot.values,   trainer.predict(X_oot)),
}

import numpy as np
label_col_data = {
    '训练集': (y_train.values > 0).astype(int),
    '验证集': (y_test.values  > 0).astype(int),
    'OOT':   (y_oot.values   > 0).astype(int),
}

reg_result = reporter.regression_report(
    datasets_reg,
    filename='LightGBM_回归模型评估报告',
    n_bins=10,
    month_col_data=month_data,
    label_col_data=label_col_data,
    feature_importance=fi_df,
    raw_df=df,
    target_col='y_reg',
    gain_score_col='gain_score', gain_label_col='overdue_30',
    scorecard_cols=['score_A', 'score_B'],
    scorecard_label_cols=['overdue_30', 'overdue_60'],
)
display(reg_result['summary'])
print('Excel:', reg_result['excel_path'])
print('HTML: ', reg_result['html_path'])

In [ ]:
print('=== 真实值分桶（三集合，训练集切点）===')
display(reg_result['sheets']['真实值分桶'])

print('=== 预测值分桶（三集合，训练集切点）===')
display(reg_result['sheets']['预测值分桶'])

print('=== 分桶矩阵（三集合合并，含 MAPE + 样本数/占比）===')
display(reg_result['sheets']['分桶矩阵'])

In [ ]:
# 自定义切点分桶
custom_bins      = [-3, -1, 0, 1, 3]
custom_bins_pred = [-3, -1, 0, 1, 3]

reg_result_custom = reporter.regression_report(
    datasets_reg,
    filename='LightGBM_回归评估_自定义分桶',
    bins=custom_bins,
    bins_pred=custom_bins_pred,
)
print('=== 自定义切点 — 真实值分桶 ===')
display(reg_result_custom['sheets']['真实值分桶'])
print('=== 自定义切点 — 预测值分桶 ===')
display(reg_result_custom['sheets']['预测值分桶'])
print('=== 自定义切点 — 分桶矩阵 ===')
display(reg_result_custom['sheets']['分桶矩阵'])

## 6. 特征分析报告

In [ ]:
analysis_path = reporter.feature_analysis_report(
    df=df,
    features=feature_cols,
    target_col='y_reg',
    dataset_col='dataset',
    month_col='month',
    filename='特征分析报告',
)
print('特征分析报告:', analysis_path)
import openpyxl
wb = openpyxl.load_workbook(analysis_path)
print('sheets:', wb.sheetnames)

## 7. 模型保存与加载

训练时传 `save_dir` 自动保存，目录结构如下：

- `01_feature_analysis/` — 特征分析结果
- `02_feature_selection/` — 特征筛选报告
- `03_model_tuning/tuning_log.xlsx` — 调参日志
- `04_model_report/` — 评估报告（Excel + HTML）
- `05_model_deploy/model.pkl` — joblib 格式，含模型对象、入模特征、最优参数

支持两种加载方式：
1. `ModelTrainer.load()` — 通过 ml_tool 加载，保持完整接口
2. `joblib.load()` — 直接加载原生模型对象，适合生产打分

In [ ]:
import os, joblib

# 查看各子目录
print('OUTPUT_DIR 子目录:')
for entry in sorted(os.scandir(OUTPUT_DIR), key=lambda e: e.name):
    if entry.is_dir():
        files = [f.name for f in os.scandir(entry.path)]
        print(f'  {entry.name}/', files)

# 方式一：ModelTrainer.load（保持完整接口）
loaded = ModelTrainer.load(OUTPUT_DIR)
pred_via_trainer = loaded.predict(X_oot)
print('\n[方式一] 入模特征:', loaded.selected_features)
print('[方式一] 预测前5个值:', pred_via_trainer[:5].round(4))

# 方式二：joblib 直接加载原生模型，适合生产环境打分
payload  = joblib.load(os.path.join(OUTPUT_DIR, '05_model_deploy', 'model.pkl'))
model    = payload['model']
features = payload['selected_features']
best_p   = payload['best_params']

print('\n[方式二] model_type:', payload['model_type'])
print('[方式二] 入模特征:', features)
print('[方式二] 最优参数（部分）:', {k: best_p[k] for k in list(best_p)[:3]})

# LightGBM Booster 直接调用 predict
pred_direct = model.predict(X_oot[features])
print('[方式二] 预测前5个值:', pred_direct[:5].round(4))

# 验证两种方式结果完全一致
max_diff = float(np.abs(pred_via_trainer - pred_direct).max())
print('两种方式最大偏差:', round(max_diff, 8), '（应为 0）')
assert max_diff == 0.0
